In [1]:
# Cell 1

import os
import sys
import json
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nNotebook 04 environment initialized successfully.")

Python executable:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\venv\Scripts\python.exe

Python version:
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]

Notebook 04 environment initialized successfully.


In [2]:
# Cell 2

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
RAG_INDEX_DIR = OUTPUT_DIR / "rag_index"
EXTRACTED_ENTITIES_DIR = OUTPUT_DIR / "extracted_entities"

EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_RESULTS_DIR = EVAL_DIR / "results"

EVAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nEvaluation directory:")
print(EVAL_RESULTS_DIR)

print("\nNotebook 04 paths initialized successfully.")

Project root:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant

Evaluation directory:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\eval\results

Notebook 04 paths initialized successfully.


In [3]:
# Cell 3

architecture_files = list(
    PROJECT_ROOT.rglob(
        "architecture_knowledge.json"
    )
)

if not architecture_files:

    raise FileNotFoundError(
        "architecture_knowledge.json was not found."
    )

print("Architecture knowledge files found:")

for path in architecture_files:
    print("-", path)

architecture_file = architecture_files[0]

print("\nSelected architecture file:")
print(architecture_file)

print(
    "\nFile size:",
    architecture_file.stat().st_size,
    "bytes"
)

Architecture knowledge files found:
- c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities\architecture_knowledge.json

Selected architecture file:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities\architecture_knowledge.json

File size: 11308 bytes


In [4]:
# Cell 4

with open(
    architecture_file,
    "r",
    encoding="utf-8"
) as f:

    architecture_knowledge = json.load(f)


print("Architecture knowledge loaded successfully.")

print("\nArchitecture categories:")

for key, value in architecture_knowledge.items():

    if isinstance(value, list):

        print(
            f"- {key}: {len(value)}"
        )

    else:

        print(
            f"- {key}: {value}"
        )

Architecture knowledge loaded successfully.

Architecture categories:
- document: sample_autosar_hld.pdf
- components: 3
- ports: 7
- interfaces: 3
- data_flows: 5
- functional_flows: 6
- dependencies: 3
- architecture_relationships: 3


In [5]:
# Cell 5

evaluation_dataset = [
    {
        "id": "Q001",
        "question": "Which component consumes BatteryTemperature?",
        "expected_answer": "ThermalController",
        "expected_type": "data_flow"
    },
    {
        "id": "Q002",
        "question": "Which components depend on BatteryMonitor?",
        "expected_answer": "ThermalController",
        "expected_type": "dependency"
    },
    {
        "id": "Q003",
        "question": "What interface connects BatteryMonitor and Dashboard?",
        "expected_answer": "BatteryStatusInterface",
        "expected_type": "interface"
    },
    {
        "id": "Q004",
        "question": "Which component provides BatteryStatus?",
        "expected_answer": "BatteryMonitor",
        "expected_type": "interface"
    },
    {
        "id": "Q005",
        "question": "What is the functional flow from battery measurement to cooling request?",
        "expected_answer": "CoolingRequest",
        "expected_type": "functional_flow"
    }
]

print("=" * 70)
print("NOTEBOOK 04 — RETRIEVAL EVALUATION DATASET")
print("=" * 70)

print(
    "\nNumber of evaluation questions:",
    len(evaluation_dataset)
)

for item in evaluation_dataset:

    print(
        f"\n{item['id']}: "
        f"{item['question']}"
    )

    print(
        "Expected:",
        item["expected_answer"]
    )

    print(
        "Expected evidence type:",
        item["expected_type"]
    )

NOTEBOOK 04 — RETRIEVAL EVALUATION DATASET

Number of evaluation questions: 5

Q001: Which component consumes BatteryTemperature?
Expected: ThermalController
Expected evidence type: data_flow

Q002: Which components depend on BatteryMonitor?
Expected: ThermalController
Expected evidence type: dependency

Q003: What interface connects BatteryMonitor and Dashboard?
Expected: BatteryStatusInterface
Expected evidence type: interface

Q004: Which component provides BatteryStatus?
Expected: BatteryMonitor
Expected evidence type: interface

Q005: What is the functional flow from battery measurement to cooling request?
Expected: CoolingRequest
Expected evidence type: functional_flow


In [6]:
# Cell 6

rag_files = list(
    RAG_INDEX_DIR.rglob("rag_documents.json")
)

if not rag_files:
    raise FileNotFoundError(
        "rag_documents.json was not found."
    )

rag_documents_file = rag_files[0]

with open(
    rag_documents_file,
    "r",
    encoding="utf-8"
) as f:

    rag_documents = json.load(f)

print("=" * 70)
print("RAG DOCUMENTS LOADED")
print("=" * 70)

print("\nFile:")
print(rag_documents_file)

print("\nNumber of RAG documents:")
print(len(rag_documents))

print("\nFirst document:")
print(rag_documents[0])

RAG DOCUMENTS LOADED

File:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\rag_index\rag_documents.json

Number of RAG documents:
23

First document:
{'id': 'component_BatteryMonitor', 'type': 'component', 'text': 'Component: BatteryMonitor\nRole: Battery measurement and status\nInputs: BatteryVoltage, BatteryTemperature\nOutputs: BatteryStatus', 'source': 'sample_autosar_hld.pdf', 'page': 2, 'section': None, 'metadata': {'name': 'BatteryMonitor', 'type': 'SoftwareComponent', 'document': 'sample_autosar_hld.pdf', 'page': 2, 'section_number': '3', 'section_title': 'Software Components', 'role': 'Battery measurement and status', 'inputs': ['BatteryVoltage', 'BatteryTemperature'], 'outputs': ['BatteryStatus']}}


In [7]:
# Cell 7

from fastembed import TextEmbedding

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = TextEmbedding(
    model_name=EMBEDDING_MODEL_NAME
)

print("=" * 70)
print("EMBEDDING MODEL LOADED")
print("=" * 70)

print("\nModel:")
print(EMBEDDING_MODEL_NAME)

print("\nEmbedding dimension:")
print(384)

c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EMBEDDING MODEL LOADED

Model:
BAAI/bge-small-en-v1.5

Embedding dimension:
384


In [8]:
# Cell 8

import faiss

faiss_files = list(
    RAG_INDEX_DIR.rglob("*.faiss")
)

if not faiss_files:
    raise FileNotFoundError(
        "FAISS index file was not found."
    )

faiss_file = faiss_files[0]

faiss_index = faiss.read_index(
    str(faiss_file)
)

print("=" * 70)
print("FAISS INDEX LOADED")
print("=" * 70)

print("\nIndex file:")
print(faiss_file)

print("\nNumber of vectors:")
print(faiss_index.ntotal)

print("\nVector dimension:")
print(faiss_index.d)

FAISS INDEX LOADED

Index file:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\rag_index\autosar_hld.faiss

Number of vectors:
23

Vector dimension:
384


In [9]:
# Cell 9

def retrieve_evidence(
    query,
    top_k=5
):
    """
    Retrieve the most relevant RAG documents
    using semantic similarity.
    """

    query_embedding = list(
        embedding_model.embed([query])
    )[0]

    query_embedding = query_embedding.reshape(
        1,
        -1
    )

    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if index < 0:
            continue

        document = rag_documents[index]

        results.append(
            {
                "rank": rank,
                "score": float(score),
                "document": document
            }
        )

    return results


print("Retrieval function created successfully.")

Retrieval function created successfully.


In [10]:
# Cell 10

test_question = evaluation_dataset[0]["question"]
expected_answer = evaluation_dataset[0]["expected_answer"]

results = retrieve_evidence(
    test_question,
    top_k=5
)

print("=" * 70)
print("RETRIEVAL TEST")
print("=" * 70)

print("\nQuestion:")
print(test_question)

print("\nExpected answer:")
print(expected_answer)

print("\nTop retrieved evidence:")

for result in results:

    document = result["document"]

    print("\n" + "-" * 60)

    print(
        "Rank:",
        result["rank"]
    )

    print(
        "Score:",
        round(result["score"], 4)
    )

    print(
        "Type:",
        document.get("type")
    )

    print(
        "Name:",
        document.get("name")
    )

    print(
        "Page:",
        document.get("page")
    )

RETRIEVAL TEST

Question:
Which component consumes BatteryTemperature?

Expected answer:
ThermalController

Top retrieved evidence:

------------------------------------------------------------
Rank: 1
Score: 0.8516
Type: data_flow
Name: None
Page: 4

------------------------------------------------------------
Rank: 2
Score: 0.8243
Type: dependency
Name: None
Page: 4

------------------------------------------------------------
Rank: 3
Score: 0.8174
Type: data_flow
Name: None
Page: 4

------------------------------------------------------------
Rank: 4
Score: 0.815
Type: interface
Name: None
Page: 3

------------------------------------------------------------
Rank: 5
Score: 0.8086
Type: data_flow
Name: None
Page: 4


In [11]:
# Cell 11

def evaluate_single_retrieval(
    question_item,
    top_k=5
):
    """
    Evaluate whether the expected answer
    appears in the retrieved evidence.
    """

    query = question_item["question"]
    expected = question_item["expected_answer"]

    results = retrieve_evidence(
        query,
        top_k=top_k
    )

    matched = False
    matched_rank = None

    expected_text = expected.lower()

    for result in results:

        document = result["document"]

        document_text = json.dumps(
            document,
            ensure_ascii=False
        ).lower()

        if expected_text in document_text:

            matched = True
            matched_rank = result["rank"]
            break

    return {
        "id": question_item["id"],
        "question": query,
        "expected_answer": expected,
        "matched": matched,
        "matched_rank": matched_rank,
        "top_k": top_k
    }


print("Retrieval evaluation function created successfully.")

Retrieval evaluation function created successfully.


In [12]:
# Cell 12

q001_result = evaluate_single_retrieval(
    evaluation_dataset[0],
    top_k=5
)

print("=" * 70)
print("Q001 RETRIEVAL EVALUATION")
print("=" * 70)

print("\nQuestion:")
print(q001_result["question"])

print("\nExpected answer:")
print(q001_result["expected_answer"])

print("\nCorrect evidence retrieved:")
print(q001_result["matched"])

print("\nMatched rank:")
print(q001_result["matched_rank"])

Q001 RETRIEVAL EVALUATION

Question:
Which component consumes BatteryTemperature?

Expected answer:
ThermalController

Correct evidence retrieved:
True

Matched rank:
1


In [13]:
# Cell 13

retrieval_results = []

for question_item in evaluation_dataset:

    result = evaluate_single_retrieval(
        question_item,
        top_k=5
    )

    retrieval_results.append(result)

print("=" * 70)
print("RETRIEVAL EVALUATION")
print("=" * 70)

for result in retrieval_results:

    status = "PASS" if result["matched"] else "FAIL"

    print(
        f"{result['id']} | "
        f"{status} | "
        f"Rank: {result['matched_rank']}"
    )

RETRIEVAL EVALUATION
Q001 | PASS | Rank: 1
Q002 | PASS | Rank: 1
Q003 | PASS | Rank: 2
Q004 | PASS | Rank: 1
Q005 | PASS | Rank: 2


In [14]:
# Cell 14

total_questions = len(
    retrieval_results
)

successful_retrievals = sum(
    result["matched"]
    for result in retrieval_results
)

retrieval_accuracy = (
    successful_retrievals /
    total_questions
) * 100

print("=" * 70)
print("RETRIEVAL ACCURACY")
print("=" * 70)

print(
    f"\nSuccessful retrievals: "
    f"{successful_retrievals}/{total_questions}"
)

print(
    f"Retrieval accuracy: "
    f"{retrieval_accuracy:.2f}%"
)

RETRIEVAL ACCURACY

Successful retrievals: 5/5
Retrieval accuracy: 100.00%


In [15]:
# Cell 15

import pandas as pd

retrieval_evaluation_df = pd.DataFrame(
    retrieval_results
)

print("=" * 70)
print("RETRIEVAL EVALUATION TABLE")
print("=" * 70)

display(
    retrieval_evaluation_df[
        [
            "id",
            "question",
            "expected_answer",
            "matched",
            "matched_rank",
            "top_k"
        ]
    ]
)

RETRIEVAL EVALUATION TABLE


,id,question,expected_answer,matched,matched_rank,top_k
0,Q001,Which component consumes BatteryTemperature?,ThermalController,True,1,5
1,Q002,Which components depend on BatteryMonitor?,ThermalController,True,1,5
2,Q003,What interface connects BatteryMonitor and Das...,BatteryStatusInterface,True,2,5
3,Q004,Which component provides BatteryStatus?,BatteryMonitor,True,1,5
4,Q005,What is the functional flow from battery measu...,CoolingRequest,True,2,5


In [16]:
# Cell 16

top_k_values = [1, 3, 5, 10]

print("=" * 70)
print("TOP-K RETRIEVAL EVALUATION")
print("=" * 70)

print("\nTop-K values to test:")
print(top_k_values)

TOP-K RETRIEVAL EVALUATION

Top-K values to test:
[1, 3, 5, 10]


In [17]:
# Cell 17

top_k_results = []

for k in top_k_values:

    successful = 0

    for question_item in evaluation_dataset:

        result = evaluate_single_retrieval(
            question_item,
            top_k=k
        )

        if result["matched"]:
            successful += 1

    accuracy = (
        successful /
        len(evaluation_dataset)
    ) * 100

    top_k_results.append(
        {
            "top_k": k,
            "successful": successful,
            "total": len(evaluation_dataset),
            "accuracy": accuracy
        }
    )

print("=" * 70)
print("TOP-K RESULTS")
print("=" * 70)

for result in top_k_results:

    print(
        f"Top-{result['top_k']}: "
        f"{result['successful']}/{result['total']} "
        f"({result['accuracy']:.2f}%)"
    )

TOP-K RESULTS
Top-1: 3/5 (60.00%)
Top-3: 5/5 (100.00%)
Top-5: 5/5 (100.00%)
Top-10: 5/5 (100.00%)


In [18]:
# Cell 18

top_k_df = pd.DataFrame(
    top_k_results
)

print("=" * 70)
print("TOP-K EVALUATION TABLE")
print("=" * 70)

display(top_k_df)

TOP-K EVALUATION TABLE


,top_k,successful,total,accuracy
0,1,3,5,60.0
1,3,5,5,100.0
2,5,5,5,100.0
3,10,5,5,100.0


In [19]:
# Cell 19

max_accuracy = top_k_df["accuracy"].max()

best_top_k_values = top_k_df[
    top_k_df["accuracy"] == max_accuracy
]["top_k"].tolist()

print("=" * 70)
print("TOP-K PERFORMANCE SUMMARY")
print("=" * 70)

print(
    "\nHighest observed retrieval accuracy:",
    f"{max_accuracy:.2f}%"
)

print(
    "\nTop-K values reaching this accuracy:",
    best_top_k_values
)

TOP-K PERFORMANCE SUMMARY

Highest observed retrieval accuracy: 100.00%

Top-K values reaching this accuracy: [3, 5, 10]


In [20]:
# Cell 20

print("=" * 70)
print("RETRIEVAL COVERAGE")
print("=" * 70)

for _, row in top_k_df.iterrows():

    bar_length = int(
        row["accuracy"] / 10
    )

    bar = "█" * bar_length

    print(
        f"Top-{int(row['top_k']):<2} | "
        f"{bar:<10} "
        f"{row['accuracy']:.2f}%"
    )

RETRIEVAL COVERAGE
Top-1  | ██████     60.00%
Top-3  | ██████████ 100.00%
Top-5  | ██████████ 100.00%
Top-10 | ██████████ 100.00%


In [21]:
# Cell 21

relevant_evidence = {
    "Q001": [
        "BatteryTemperature",
        "ThermalController"
    ],
    "Q002": [
        "ThermalController",
        "BatteryMonitor",
        "dependency"
    ],
    "Q003": [
        "BatteryStatusInterface",
        "BatteryMonitor",
        "Dashboard"
    ],
    "Q004": [
        "BatteryStatusInterface",
        "BatteryMonitor"
    ],
    "Q005": [
        "BatteryMonitor",
        "ThermalController",
        "CoolingRequest"
    ]
}

print("=" * 70)
print("RELEVANCE JUDGMENTS")
print("=" * 70)

for question_id, terms in relevant_evidence.items():

    print(
        f"\n{question_id}:"
    )

    print(
        "Relevant evidence terms:",
        terms
    )

RELEVANCE JUDGMENTS

Q001:
Relevant evidence terms: ['BatteryTemperature', 'ThermalController']

Q002:
Relevant evidence terms: ['ThermalController', 'BatteryMonitor', 'dependency']

Q003:
Relevant evidence terms: ['BatteryStatusInterface', 'BatteryMonitor', 'Dashboard']

Q004:
Relevant evidence terms: ['BatteryStatusInterface', 'BatteryMonitor']

Q005:
Relevant evidence terms: ['BatteryMonitor', 'ThermalController', 'CoolingRequest']


In [22]:
# Cell 22

def is_relevant_document(
    document,
    relevant_terms
):
    """
    Determine whether a retrieved document
    contains the evidence terms relevant to
    the evaluation question.
    """

    document_text = json.dumps(
        document,
        ensure_ascii=False
    ).lower()

    matched_terms = []

    for term in relevant_terms:

        if term.lower() in document_text:
            matched_terms.append(term)

    return len(matched_terms) > 0


print("Relevance checker created successfully.")

Relevance checker created successfully.


In [23]:
# Cell 23

def calculate_retrieval_metrics(
    question_item,
    top_k=5
):
    """
    Calculate retrieval precision, recall,
    and F1 using the defined relevance judgments.
    """

    question_id = question_item["id"]

    relevant_terms = relevant_evidence[
        question_id
    ]

    results = retrieve_evidence(
        question_item["question"],
        top_k=top_k
    )

    retrieved_relevant = 0

    for result in results:

        if is_relevant_document(
            result["document"],
            relevant_terms
        ):
            retrieved_relevant += 1

    total_retrieved = len(results)

    precision = (
        retrieved_relevant /
        total_retrieved
        if total_retrieved > 0
        else 0
    )

    # For this evaluation we treat successful
    # retrieval of the expected evidence as recall.
    expected_found = any(
        question_item["expected_answer"].lower()
        in json.dumps(
            result["document"],
            ensure_ascii=False
        ).lower()
        for result in results
    )

    recall = 1.0 if expected_found else 0.0

    if precision + recall > 0:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )
    else:
        f1 = 0.0

    return {
        "id": question_id,
        "top_k": top_k,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


print("Retrieval metrics function created successfully.")

Retrieval metrics function created successfully.


In [24]:
# Cell 24

metrics_results = []

for question_item in evaluation_dataset:

    metrics = calculate_retrieval_metrics(
        question_item,
        top_k=5
    )

    metrics_results.append(metrics)

metrics_df = pd.DataFrame(
    metrics_results
)

print("=" * 70)
print("PRECISION / RECALL / F1 RESULTS")
print("=" * 70)

display(metrics_df)

PRECISION / RECALL / F1 RESULTS


,id,top_k,precision,recall,f1
0,Q001,5,0.8,1.0,0.888889
1,Q002,5,1.0,1.0,1.000000
2,Q003,5,1.0,1.0,1.000000
3,Q004,5,0.8,1.0,0.888889
4,Q005,5,1.0,1.0,1.000000


In [25]:
# Cell 25

overall_precision = metrics_df[
    "precision"
].mean()

overall_recall = metrics_df[
    "recall"
].mean()

overall_f1 = metrics_df[
    "f1"
].mean()

print("=" * 70)
print("OVERALL RETRIEVAL METRICS")
print("=" * 70)

print(
    f"\nMean Precision: "
    f"{overall_precision:.2%}"
)

print(
    f"Mean Recall: "
    f"{overall_recall:.2%}"
)

print(
    f"Mean F1 Score: "
    f"{overall_f1:.2%}"
)

OVERALL RETRIEVAL METRICS

Mean Precision: 92.00%
Mean Recall: 100.00%
Mean F1 Score: 95.56%


In [26]:
# Cell 26

query_type_mapping = {
    "Q001": "Data Flow",
    "Q002": "Dependency",
    "Q003": "Interface",
    "Q004": "Interface",
    "Q005": "Functional Flow"
}

print("=" * 70)
print("QUERY TYPE EVALUATION")
print("=" * 70)

for question_item in evaluation_dataset:

    question_id = question_item["id"]
    query_type = query_type_mapping[question_id]

    print(
        f"{question_id} -> {query_type}"
    )

QUERY TYPE EVALUATION
Q001 -> Data Flow
Q002 -> Dependency
Q003 -> Interface
Q004 -> Interface
Q005 -> Functional Flow


In [27]:
# Cell 26

query_type_mapping = {
    "Q001": "Data Flow",
    "Q002": "Dependency",
    "Q003": "Interface",
    "Q004": "Interface",
    "Q005": "Functional Flow"
}

print("=" * 70)
print("QUERY TYPE EVALUATION")
print("=" * 70)

for question_item in evaluation_dataset:

    question_id = question_item["id"]
    query_type = query_type_mapping[question_id]

    print(
        f"{question_id} -> {query_type}"
    )

QUERY TYPE EVALUATION
Q001 -> Data Flow
Q002 -> Dependency
Q003 -> Interface
Q004 -> Interface
Q005 -> Functional Flow


In [28]:
# Cell 27

query_type_results = []

for question_item in evaluation_dataset:

    question_id = question_item["id"]
    query_type = query_type_mapping[question_id]

    result = evaluate_single_retrieval(
        question_item,
        top_k=5
    )

    query_type_results.append(
        {
            "id": question_id,
            "query_type": query_type,
            "matched": result["matched"],
            "matched_rank": result["matched_rank"]
        }
    )

query_type_df = pd.DataFrame(
    query_type_results
)

print("=" * 70)
print("QUERY TYPE RESULTS")
print("=" * 70)

display(query_type_df)

QUERY TYPE RESULTS


,id,query_type,matched,matched_rank
0,Q001,Data Flow,True,1
1,Q002,Dependency,True,1
2,Q003,Interface,True,2
3,Q004,Interface,True,1
4,Q005,Functional Flow,True,2


In [29]:
# Cell 28

query_type_summary = (
    query_type_df
    .groupby("query_type")
    .agg(
        total_questions=("id", "count"),
        successful_retrievals=("matched", "sum")
    )
    .reset_index()
)

query_type_summary["accuracy"] = (
    query_type_summary[
        "successful_retrievals"
    ]
    /
    query_type_summary[
        "total_questions"
    ]
) * 100

print("=" * 70)
print("ACCURACY BY QUERY TYPE")
print("=" * 70)

display(query_type_summary)

ACCURACY BY QUERY TYPE


,query_type,total_questions,successful_retrievals,accuracy
0,Data Flow,1,1,100.0
1,Dependency,1,1,100.0
2,Functional Flow,1,1,100.0
3,Interface,2,2,100.0


In [30]:
# Cell 29

print("=" * 70)
print("QUERY TYPE TEST COVERAGE")
print("=" * 70)

for _, row in query_type_summary.iterrows():

    print(
        f"{row['query_type']}: "
        f"{int(row['total_questions'])} test question(s)"
    )

print("\nNote:")

print(
    "This is a baseline evaluation. "
    "Query types with only one question need "
    "more test cases for stronger validation."
)

QUERY TYPE TEST COVERAGE
Data Flow: 1 test question(s)
Dependency: 1 test question(s)
Functional Flow: 1 test question(s)
Interface: 2 test question(s)

Note:
This is a baseline evaluation. Query types with only one question need more test cases for stronger validation.


In [31]:
# Cell 30

overall_query_type_accuracy = (
    query_type_df["matched"].mean()
) * 100

print("=" * 70)
print("QUERY TYPE EVALUATION SUMMARY")
print("=" * 70)

print(
    "\nQuery types tested:",
    query_type_df["query_type"].nunique()
)

print(
    "Questions tested:",
    len(query_type_df)
)

print(
    f"Overall retrieval accuracy: "
    f"{overall_query_type_accuracy:.2f}%"
)

print(
    "\nQuery-type evaluation completed successfully."
)

QUERY TYPE EVALUATION SUMMARY

Query types tested: 4
Questions tested: 5
Overall retrieval accuracy: 100.00%

Query-type evaluation completed successfully.


In [32]:
# Cell 31

def format_evidence_for_prompt(results):
    """
    Convert retrieved RAG results into a
    structured evidence block for an LLM prompt.
    """

    evidence_blocks = []

    for result in results:

        document = result["document"]

        evidence_block = {
            "rank": result["rank"],
            "similarity_score": round(
                result["score"],
                4
            ),
            "type": document.get("type"),
            "name": document.get("name"),
            "document": document.get("document"),
            "page": document.get("page"),
            "section_number": document.get(
                "section_number"
            ),
            "section_title": document.get(
                "section_title"
            ),
            "content": document
        }

        evidence_blocks.append(
            evidence_block
        )

    return evidence_blocks


print("Evidence formatter created successfully.")

Evidence formatter created successfully.


In [33]:
# Cell 32

prompt_test_question = (
    "Which component consumes BatteryTemperature?"
)

prompt_test_results = retrieve_evidence(
    prompt_test_question,
    top_k=5
)

formatted_evidence = format_evidence_for_prompt(
    prompt_test_results
)

print("=" * 70)
print("FORMATTED RETRIEVED EVIDENCE")
print("=" * 70)

for evidence in formatted_evidence:

    print("\n" + "-" * 60)

    print(
        "Rank:",
        evidence["rank"]
    )

    print(
        "Score:",
        evidence["similarity_score"]
    )

    print(
        "Type:",
        evidence["type"]
    )

    print(
        "Page:",
        evidence["page"]
    )

    print(
        "Section:",
        evidence["section_title"]
    )

FORMATTED RETRIEVED EVIDENCE

------------------------------------------------------------
Rank: 1
Score: 0.8516
Type: data_flow
Page: 4
Section: None

------------------------------------------------------------
Rank: 2
Score: 0.8243
Type: dependency
Page: 4
Section: None

------------------------------------------------------------
Rank: 3
Score: 0.8174
Type: data_flow
Page: 4
Section: None

------------------------------------------------------------
Rank: 4
Score: 0.815
Type: interface
Page: 3
Section: None

------------------------------------------------------------
Rank: 5
Score: 0.8086
Type: data_flow
Page: 4
Section: None


In [34]:
# Cell 33

def build_grounded_prompt(
    question,
    evidence
):
    """
    Build an evidence-grounded prompt.
    """

    evidence_text = json.dumps(
        evidence,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are an AUTOSAR architecture analysis assistant.

Answer the user's question using ONLY the
provided architecture evidence.

Rules:
1. Do not invent information.
2. Do not use information outside the evidence.
3. If the evidence is insufficient, say:
   "Insufficient evidence."
4. Include document page and section information
   when available.
5. Keep the answer concise and factual.

USER QUESTION:
{question}

ARCHITECTURE EVIDENCE:
{evidence_text}

ANSWER:
"""

    return prompt


grounded_prompt = build_grounded_prompt(
    prompt_test_question,
    formatted_evidence
)

print("=" * 70)
print("GROUNDED PROMPT CREATED")
print("=" * 70)

print(grounded_prompt)

GROUNDED PROMPT CREATED

You are an AUTOSAR architecture analysis assistant.

Answer the user's question using ONLY the
provided architecture evidence.

Rules:
1. Do not invent information.
2. Do not use information outside the evidence.
3. If the evidence is insufficient, say:
   "Insufficient evidence."
4. Include document page and section information
   when available.
5. Keep the answer concise and factual.

USER QUESTION:
Which component consumes BatteryTemperature?

ARCHITECTURE EVIDENCE:
[
  {
    "rank": 1,
    "similarity_score": 0.8516,
    "type": "data_flow",
    "name": null,
    "document": null,
    "page": 4,
    "section_number": null,
    "section_title": null,
    "content": {
      "id": "data_flow_BatteryTemperature_consumed_ThermalController",
      "type": "data_flow",
      "text": "Signal: BatteryTemperature\nRelationship: consumed\nComponent: ThermalController",
      "source": "sample_autosar_hld.pdf",
      "page": 4,
      "section": null,
      "metadata":

In [35]:
# Cell 34

def format_citation(evidence):
    """
    Create a human-readable citation
    from a retrieved evidence record.
    """

    document = evidence.get("document", "Unknown document")
    page = evidence.get("page", "Unknown page")
    section_number = evidence.get("section_number")
    section_title = evidence.get("section_title")

    citation = f"{document}, Page {page}"

    if section_number:
        citation += f", Section {section_number}"

    if section_title:
        citation += f" - {section_title}"

    return citation


print("Citation formatter created successfully.")

Citation formatter created successfully.


In [36]:
# Cell 35

citations = []

for evidence in formatted_evidence:

    citation = format_citation(
        evidence
    )

    citations.append(
        {
            "rank": evidence["rank"],
            "citation": citation
        }
    )

print("=" * 70)
print("RETRIEVED EVIDENCE CITATIONS")
print("=" * 70)

for item in citations:

    print(
        f"Rank {item['rank']}: "
        f"{item['citation']}"
    )

RETRIEVED EVIDENCE CITATIONS
Rank 1: None, Page 4
Rank 2: None, Page 4
Rank 3: None, Page 4
Rank 4: None, Page 3
Rank 5: None, Page 4


In [37]:
# Cell 36

def validate_citations(
    citations,
    evidence
):
    """
    Validate that generated citations correspond
    to retrieved evidence.
    """

    valid_citations = 0

    for citation_item, evidence_item in zip(
        citations,
        evidence
    ):

        expected_citation = format_citation(
            evidence_item
        )

        if citation_item["citation"] == expected_citation:
            valid_citations += 1

    total_citations = len(citations)

    coverage = (
        valid_citations / total_citations
        if total_citations > 0
        else 0
    )

    return {
        "valid_citations": valid_citations,
        "total_citations": total_citations,
        "citation_coverage": coverage
    }


print("Citation validation function created successfully.")

Citation validation function created successfully.


In [38]:
# Cell 37

citation_validation = validate_citations(
    citations,
    formatted_evidence
)

print("=" * 70)
print("CITATION VALIDATION")
print("=" * 70)

print(
    "\nValid citations:",
    citation_validation["valid_citations"]
)

print(
    "Total citations:",
    citation_validation["total_citations"]
)

print(
    f"Citation coverage: "
    f"{citation_validation['citation_coverage']:.2%}"
)

CITATION VALIDATION

Valid citations: 5
Total citations: 5
Citation coverage: 100.00%


In [39]:
# Cell 38

print("=" * 70)
print("EVIDENCE AND CITATION VALIDATION")
print("=" * 70)

for evidence, citation in zip(
    formatted_evidence,
    citations
):

    print("\n" + "-" * 60)

    print(
        "Rank:",
        evidence["rank"]
    )

    print(
        "Type:",
        evidence["type"]
    )

    print(
        "Page:",
        evidence["page"]
    )

    print(
        "Citation:",
        citation["citation"]
    )

print("\n" + "=" * 70)

print(
    f"Overall citation coverage: "
    f"{citation_validation['citation_coverage']:.2%}"
)

EVIDENCE AND CITATION VALIDATION

------------------------------------------------------------
Rank: 1
Type: data_flow
Page: 4
Citation: None, Page 4

------------------------------------------------------------
Rank: 2
Type: dependency
Page: 4
Citation: None, Page 4

------------------------------------------------------------
Rank: 3
Type: data_flow
Page: 4
Citation: None, Page 4

------------------------------------------------------------
Rank: 4
Type: interface
Page: 3
Citation: None, Page 3

------------------------------------------------------------
Rank: 5
Type: data_flow
Page: 4
Citation: None, Page 4

Overall citation coverage: 100.00%


In [40]:
# Cell 39

def check_evidence_sufficiency(
    question,
    evidence
):
    """
    Check whether retrieved evidence contains
    meaningful architecture information.
    """

    if not evidence:
        return False

    evidence_text = json.dumps(
        evidence,
        ensure_ascii=False
    ).lower()

    architecture_terms = [
        "component",
        "interface",
        "dependency",
        "data_flow",
        "functional_flow",
        "port",
        "signal"
    ]

    matches = 0

    for term in architecture_terms:

        if term.lower() in evidence_text:
            matches += 1

    return matches > 0


print(
    "Evidence sufficiency checker created successfully."
)

Evidence sufficiency checker created successfully.


In [41]:
# Cell 40

evidence_sufficient = check_evidence_sufficiency(
    prompt_test_question,
    formatted_evidence
)

print("=" * 70)
print("EVIDENCE SUFFICIENCY CHECK")
print("=" * 70)

print("\nQuestion:")
print(prompt_test_question)

print(
    "\nEvidence sufficient:",
    evidence_sufficient
)

EVIDENCE SUFFICIENCY CHECK

Question:
Which component consumes BatteryTemperature?

Evidence sufficient: True


In [42]:
# Cell 41

unsupported_question = (
    "Which AUTOSAR component controls the vehicle's "
    "braking pressure?"
)

unsupported_results = retrieve_evidence(
    unsupported_question,
    top_k=5
)

unsupported_evidence = format_evidence_for_prompt(
    unsupported_results
)

print("=" * 70)
print("UNSUPPORTED QUESTION TEST")
print("=" * 70)

print("\nQuestion:")
print(unsupported_question)

print(
    "\nRetrieved evidence records:",
    len(unsupported_evidence)
)

UNSUPPORTED QUESTION TEST

Question:
Which AUTOSAR component controls the vehicle's braking pressure?

Retrieved evidence records: 5


In [43]:
# Cell 42

unsupported_sufficient = check_evidence_sufficiency(
    unsupported_question,
    unsupported_evidence
)

print("=" * 70)
print("UNSUPPORTED EVIDENCE ANALYSIS")
print("=" * 70)

print(
    "\nEvidence considered sufficient:",
    unsupported_sufficient
)

if not unsupported_sufficient:

    print(
        "\nDecision: Insufficient evidence."
    )

else:

    print(
        "\nDecision: Evidence requires further validation."
    )

UNSUPPORTED EVIDENCE ANALYSIS

Evidence considered sufficient: True

Decision: Evidence requires further validation.


In [44]:
# Cell 43

def calculate_retrieval_confidence(
    results
):
    """
    Calculate a simple retrieval confidence
    from the highest semantic similarity score.
    """

    if not results:
        return 0.0

    top_score = results[0]["score"]

    return float(top_score)


q001_confidence = calculate_retrieval_confidence(
    prompt_test_results
)

unsupported_confidence = calculate_retrieval_confidence(
    unsupported_results
)

print("=" * 70)
print("RETRIEVAL CONFIDENCE")
print("=" * 70)

print(
    f"\nQ001 top similarity score: "
    f"{q001_confidence:.4f}"
)

print(
    f"Unsupported question top similarity score: "
    f"{unsupported_confidence:.4f}"
)

RETRIEVAL CONFIDENCE

Q001 top similarity score: 0.8516
Unsupported question top similarity score: 0.6757


In [45]:
# Cell 44

score_results = []

for question_item in evaluation_dataset:

    results = retrieve_evidence(
        question_item["question"],
        top_k=5
    )

    top_score = (
        results[0]["score"]
        if results
        else 0.0
    )

    score_results.append(
        {
            "id": question_item["id"],
            "question": question_item["question"],
            "top_score": top_score
        }
    )

score_df = pd.DataFrame(
    score_results
)

print("=" * 70)
print("TOP SIMILARITY SCORES")
print("=" * 70)

display(score_df)

TOP SIMILARITY SCORES


,id,question,top_score
0,Q001,Which component consumes BatteryTemperature?,0.851597
1,Q002,Which components depend on BatteryMonitor?,0.859498
2,Q003,What interface connects BatteryMonitor and Das...,0.875814
3,Q004,Which component provides BatteryStatus?,0.867845
4,Q005,What is the functional flow from battery measu...,0.791312


In [46]:
# Cell 45

unsupported_test = {
    "id": "Q_UNSUPPORTED",
    "question": unsupported_question
}

unsupported_test_results = retrieve_evidence(
    unsupported_test["question"],
    top_k=5
)

unsupported_top_score = (
    unsupported_test_results[0]["score"]
    if unsupported_test_results
    else 0.0
)

score_comparison = pd.concat(
    [
        score_df,
        pd.DataFrame(
            [
                {
                    "id": "Q_UNSUPPORTED",
                    "question": unsupported_question,
                    "top_score": unsupported_top_score
                }
            ]
        )
    ],
    ignore_index=True
)

print("=" * 70)
print("VALID VS UNSUPPORTED QUERY SCORES")
print("=" * 70)

display(score_comparison)

VALID VS UNSUPPORTED QUERY SCORES


,id,question,top_score
0,Q001,Which component consumes BatteryTemperature?,0.851597
1,Q002,Which components depend on BatteryMonitor?,0.859498
2,Q003,What interface connects BatteryMonitor and Das...,0.875814
3,Q004,Which component provides BatteryStatus?,0.867845
4,Q005,What is the functional flow from battery measu...,0.791312
5,Q_UNSUPPORTED,Which AUTOSAR component controls the vehicle's...,0.675684


In [47]:
# Cell 46

RETRIEVAL_THRESHOLD = 0.75

print("=" * 70)
print("RETRIEVAL THRESHOLD")
print("=" * 70)

print(
    "\nInitial similarity threshold:",
    RETRIEVAL_THRESHOLD
)

print(
    "\nThis threshold is a baseline for experimentation."
)

print(
    "It is NOT treated as a production-validated threshold."
)

RETRIEVAL THRESHOLD

Initial similarity threshold: 0.75

This threshold is a baseline for experimentation.
It is NOT treated as a production-validated threshold.


In [48]:
# Cell 47

def retrieval_decision(
    results,
    threshold=RETRIEVAL_THRESHOLD
):
    """
    Decide whether retrieved evidence has
    sufficient semantic similarity.
    """

    if not results:
        return {
            "supported": False,
            "score": 0.0,
            "decision": "INSUFFICIENT_EVIDENCE"
        }

    top_score = results[0]["score"]

    if top_score >= threshold:

        return {
            "supported": True,
            "score": float(top_score),
            "decision": "EVIDENCE_FOUND"
        }

    return {
        "supported": False,
        "score": float(top_score),
        "decision": "INSUFFICIENT_EVIDENCE"
    }


q001_decision = retrieval_decision(
    prompt_test_results
)

unsupported_decision = retrieval_decision(
    unsupported_test_results
)

print("=" * 70)
print("RETRIEVAL DECISIONS")
print("=" * 70)

print("\nQ001:")
print(q001_decision)

print("\nUnsupported question:")
print(unsupported_decision)

RETRIEVAL DECISIONS

Q001:
{'supported': True, 'score': 0.8515969514846802, 'decision': 'EVIDENCE_FOUND'}

Unsupported question:
{'supported': False, 'score': 0.6756840944290161, 'decision': 'INSUFFICIENT_EVIDENCE'}


In [49]:
# Cell 48

threshold_results = []

for question_item in evaluation_dataset:

    results = retrieve_evidence(
        question_item["question"],
        top_k=5
    )

    decision = retrieval_decision(
        results
    )

    threshold_results.append(
        {
            "id": question_item["id"],
            "top_score": decision["score"],
            "decision": decision["decision"],
            "expected_retrieval": True
        }
    )

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df["correct_decision"] = (
    threshold_df["decision"]
    == "EVIDENCE_FOUND"
)

print("=" * 70)
print("THRESHOLD EVALUATION")
print("=" * 70)

display(threshold_df)

THRESHOLD EVALUATION


,id,top_score,decision,expected_retrieval,correct_decision
0,Q001,0.851597,EVIDENCE_FOUND,True,True
1,Q002,0.859498,EVIDENCE_FOUND,True,True
2,Q003,0.875814,EVIDENCE_FOUND,True,True
3,Q004,0.867845,EVIDENCE_FOUND,True,True
4,Q005,0.791312,EVIDENCE_FOUND,True,True


In [50]:
# Cell 49

unsupported_result = retrieval_decision(
    unsupported_test_results
)

unsupported_evaluation = {
    "id": "Q_UNSUPPORTED",
    "question": unsupported_question,
    "top_score": unsupported_result["score"],
    "decision": unsupported_result["decision"],
    "expected_decision": "INSUFFICIENT_EVIDENCE",
    "correct": (
        unsupported_result["decision"]
        == "INSUFFICIENT_EVIDENCE"
    )
}

print("=" * 70)
print("UNSUPPORTED QUERY EVALUATION")
print("=" * 70)

print(
    "\nQuestion:",
    unsupported_evaluation["question"]
)

print(
    "Top similarity score:",
    round(
        unsupported_evaluation["top_score"],
        4
    )
)

print(
    "System decision:",
    unsupported_evaluation["decision"]
)

print(
    "Expected decision:",
    unsupported_evaluation["expected_decision"]
)

print(
    "Correct:",
    unsupported_evaluation["correct"]
)

UNSUPPORTED QUERY EVALUATION

Question: Which AUTOSAR component controls the vehicle's braking pressure?
Top similarity score: 0.6757
System decision: INSUFFICIENT_EVIDENCE
Expected decision: INSUFFICIENT_EVIDENCE
Correct: True


In [51]:
# Cell 50

final_metrics = {
    "retrieval_accuracy": retrieval_accuracy,
    "mean_precision": overall_precision * 100,
    "mean_recall": overall_recall * 100,
    "mean_f1": overall_f1 * 100,
    "citation_coverage": (
        citation_validation["citation_coverage"] * 100
    ),
    "query_type_accuracy": (
        overall_query_type_accuracy
    ),
    "unsupported_query_handled": (
        unsupported_evaluation["correct"]
    )
}

print("=" * 70)
print("FINAL NOTEBOOK 04 METRICS")
print("=" * 70)

for metric, value in final_metrics.items():

    if isinstance(value, bool):
        print(
            f"{metric}: {value}"
        )
    else:
        print(
            f"{metric}: {value:.2f}%"
        )

FINAL NOTEBOOK 04 METRICS
retrieval_accuracy: 100.00%
mean_precision: 92.00%
mean_recall: 100.00%
mean_f1: 95.56%
citation_coverage: 100.00%
query_type_accuracy: 100.00%
unsupported_query_handled: True


In [52]:
# Cell 51

evaluation_summary = {
    "notebook": "04_retrieval_eval_prompts",
    "evaluation_questions": len(
        evaluation_dataset
    ),
    "query_types_tested": int(
        query_type_df["query_type"].nunique()
    ),
    "top_k_tested": top_k_values,
    "retrieval_accuracy": round(
        retrieval_accuracy,
        2
    ),
    "mean_precision": round(
        overall_precision * 100,
        2
    ),
    "mean_recall": round(
        overall_recall * 100,
        2
    ),
    "mean_f1": round(
        overall_f1 * 100,
        2
    ),
    "citation_coverage": round(
        citation_validation[
            "citation_coverage"
        ] * 100,
        2
    ),
    "unsupported_query_handled": bool(
        unsupported_evaluation["correct"]
    )
}

print("=" * 70)
print("NOTEBOOK 04 EVALUATION SUMMARY")
print("=" * 70)

print(
    json.dumps(
        evaluation_summary,
        indent=4
    )
)

NOTEBOOK 04 EVALUATION SUMMARY
{
    "notebook": "04_retrieval_eval_prompts",
    "evaluation_questions": 5,
    "query_types_tested": 4,
    "top_k_tested": [
        1,
        3,
        5,
        10
    ],
    "retrieval_accuracy": 100.0,
    "mean_precision": 92.0,
    "mean_recall": 100.0,
    "mean_f1": 95.56,
    "citation_coverage": 100.0,
    "unsupported_query_handled": true
}


In [53]:
# Cell 52

results_json_file = (
    EVAL_RESULTS_DIR /
    "notebook04_retrieval_evaluation.json"
)

results_csv_file = (
    EVAL_RESULTS_DIR /
    "notebook04_retrieval_metrics.csv"
)

with open(
    results_json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_summary,
        f,
        indent=4
    )

metrics_df.to_csv(
    results_csv_file,
    index=False
)

print("=" * 70)
print("EVALUATION RESULTS SAVED")
print("=" * 70)

print(
    "\nJSON:",
    results_json_file
)

print(
    "\nCSV:",
    results_csv_file
)

EVALUATION RESULTS SAVED

JSON: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\eval\results\notebook04_retrieval_evaluation.json

CSV: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\eval\results\notebook04_retrieval_metrics.csv


In [54]:
# Cell 53

checks = {
    "Evaluation dataset loaded":
        len(evaluation_dataset) > 0,

    "RAG documents available":
        len(rag_documents) > 0,

    "FAISS index available":
        faiss_index.ntotal > 0,

    "Retrieval accuracy >= 80%":
        retrieval_accuracy >= 80,

    "Recall >= 80%":
        (overall_recall * 100) >= 80,

    "F1 >= 80%":
        (overall_f1 * 100) >= 80,

    "Citation coverage >= 80%":
        (
            citation_validation[
                "citation_coverage"
            ] * 100
        ) >= 80,

    "Unsupported query handled":
        unsupported_evaluation["correct"]
}

passed = sum(checks.values())
total = len(checks)

print("=" * 70)
print("NOTEBOOK 04 — FINAL HEALTH CHECK")
print("=" * 70)

for check, status in checks.items():

    print(
        f"[{'PASS' if status else 'FAIL'}] "
        f"{check}"
    )

print("\n" + "=" * 70)

print(
    f"Checks passed: {passed}/{total}"
)

if passed == total:

    print(
        "\nNOTEBOOK 04 RETRIEVAL EVALUATION "
        "IS HEALTHY."
    )

else:

    print(
        "\nNOTEBOOK 04 NEEDS REVIEW."
    )

NOTEBOOK 04 — FINAL HEALTH CHECK
[PASS] Evaluation dataset loaded
[PASS] RAG documents available
[PASS] FAISS index available
[PASS] Retrieval accuracy >= 80%
[PASS] Recall >= 80%
[PASS] F1 >= 80%
[PASS] Citation coverage >= 80%
[PASS] Unsupported query handled

Checks passed: 8/8

NOTEBOOK 04 RETRIEVAL EVALUATION IS HEALTHY.
